# Module 0 lab: notebooks, NumPy and your first qubit

**Quantum Computing Foundations · Module 0 · about 40 minutes**

In this lab you will:

1. run and change cells in a Colab notebook;
2. refresh the NumPy and complex-number tools the course uses;
3. build, draw and run your first one-qubit circuit;
4. find and fix a bug;
5. rotate a qubit by your own angle and get a verification value for Canvas.

The **Module 0 lab check** in Canvas asks for results from this notebook. Keep both open side by side. Run each code cell with **Shift + Enter**, in order.

## Step 0: set up

This cell installs the exact package versions the course is tested with. It takes about a minute the first time. If Colab asks you to restart the session afterwards, choose **Restart session**, then continue from Step 1.

In [ ]:
%pip install -q qiskit==2.5.2 qiskit-aer==0.17.2 qiskit-ibm-runtime

## Step 1: how a notebook works

A notebook mixes text cells like this one with **code cells**. Running a code cell prints its result underneath. Later cells can use what earlier cells created, so run them in order.

**Try it:** change `"Ada"` to your own first name, then run the cell again.

In [ ]:
name = "Ada"
print(f"Hello, {name}! Your notebook is working.")

## Step 2: NumPy and complex numbers

Quantum states are written with **complex numbers**, numbers of the form a + bi. Python writes i as `j`, so 0.6 + 0.8i is `0.6 + 0.8j`.

Three tools you will use in every module:

| Tool | Python | Example |
|---|---|---|
| Size squared, \|z\|² | `abs(z)**2` | \|0.6 + 0.8i\|² = 0.36 + 0.64 = 1 |
| Complex conjugate, z* | `np.conj(z)` | (0.6 + 0.8i)* = 0.6 − 0.8i |
| A vector of amplitudes | `np.array([a, b])` | a state of one qubit |

Run the cell and compare its output with the table.

In [ ]:
import numpy as np

z = 0.6 + 0.8j
print("z =", z)
print("|z|^2 =", round(abs(z)**2, 6))
print("conjugate of z =", np.conj(z))

A one-qubit state is a vector of two amplitudes: the first for |0⟩, the second for |1⟩. **The probability of each measurement result is the size squared of its amplitude**, and the probabilities add up to 1.

The state below has amplitude 0.6 for |0⟩ and 0.8i for |1⟩. Run the cell. Canvas asks for the probability of measuring 1.

In [ ]:
psi = np.array([0.6, 0.8j])

probabilities = np.abs(psi)**2
print("P(0) =", round(probabilities[0], 3))
print("P(1) =", round(probabilities[1], 3))
print("Total =", round(probabilities.sum(), 3))

## Step 3: your first quantum circuit

A **circuit** is a list of operations on qubits. This one has one qubit and one classical bit to store the measurement:

- `qc.h(0)` applies the **Hadamard gate**, which puts the qubit into an equal superposition of 0 and 1;
- `qc.measure(0, 0)` measures qubit 0 into classical bit 0.

Run the cell to draw the circuit, then run the next cell to measure it 1,000 times on a simulator.

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(1, 1)
qc.h(0)
qc.measure(0, 0)
qc.draw()

In [ ]:
from qiskit import transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()
counts = sim.run(transpile(qc, sim), shots=1000).result().get_counts()
print(counts)

You should see about 500 of each result. The exact numbers change each run: measurement results are random.

**Change one thing:** in the circuit cell above, add a second line `qc.h(0)` directly after the first, so the Hadamard gate is applied **twice**. Run the circuit cell and the simulator cell again. Canvas asks how many times out of 1,000 you get the result 1. (You will see why in Module 3.)

## Step 4: find and fix the bug

The cell below has one bug. Run it and read the error message at the bottom. Then fix the line so the cell runs. Canvas asks which change fixes it.

In [ ]:
qc_bug = QuantumCircuit(1, 1)
qc_bug.h(0)
qc_bug.measure(0, 1)
print(sim.run(transpile(qc_bug, sim), shots=100).result().get_counts())

## Step 5: rotate the qubit by your own angle

Open the Canvas quiz **Module 0 lab check**. Question 1 shows your personal angle θ in radians, for example `θ = 1.25`. Type it below in place of `None` and run the cell.

In [ ]:
THETA = None   # for example: THETA = 1.25

print("Your angle:", THETA)

The gate `qc.ry(angle, 0)` rotates the qubit by an angle around the y-axis of the Bloch sphere (Module 2). Complete `build_rotation()` so it applies `ry` with **your** `THETA` to qubit 0, then measures it. Run the cell.

In [ ]:
def build_rotation():
    qc = QuantumCircuit(1, 1)

    # YOUR CODE: apply ry with your THETA to qubit 0


    qc.measure(0, 0)
    return qc

build_rotation().draw()

Now run the check cell. **Do not edit it.** If your circuit is right, it prints your verification value. Type it into Canvas question 1 with three decimal places, for example `0.315`.

In [ ]:
# CHECK CELL: do not edit
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, state_fidelity


def _strip_final_measurements(qc):
    """Remove only the last measurement on each qubit; any earlier measurement or reset stays in."""
    keep = list(qc.data)
    done = set()
    for idx in range(len(keep) - 1, -1, -1):
        inst = keep[idx]
        qubits = [qc.find_bit(q).index for q in inst.qubits]
        if inst.operation.name == "barrier":
            continue
        if inst.operation.name == "measure" and qubits[0] not in done:
            keep[idx] = None
            done.add(qubits[0])
            continue
        done.update(qubits)
        if len(done) == qc.num_qubits:
            break
    body = qc.copy_empty_like()
    for inst in keep:
        if inst is not None:
            body.append(inst)
    return body


def check_rotation(qc, theta):
    if not isinstance(qc, QuantumCircuit):
        return False, ["build_rotation() must return a QuantumCircuit."], None
    if qc.num_qubits != 1:
        return False, [f"The circuit has {qc.num_qubits} qubits; this task uses exactly 1."], None
    if not any(inst.operation.name == "measure" for inst in qc.data):
        return False, ["Add a measurement at the end: qc.measure(0, 0)."], None
    body = _strip_final_measurements(qc)
    if any(inst.operation.name in ("measure", "reset") for inst in body.data):
        return False, ["Measure the qubit only once, at the end."], None
    body.remove_final_measurements()
    state = Statevector(body)
    target = Statevector([np.cos(theta / 2), np.sin(theta / 2)])
    fid = state_fidelity(state, target)
    if fid < 0.99:
        return False, [f"The state is not Ry(THETA) applied to |0> (fidelity {fid:.3f}). "
                       "Use qc.ry(THETA, 0) with your THETA from Step 5."], None
    p1 = float(state.probabilities()[1])
    return True, [f"State check passed (fidelity {fid:.3f})."], round(p1, 3)


if THETA is None:
    print("Enter your angle THETA in Step 5 first, then run this cell again.")
else:
    ok, messages, value = check_rotation(build_rotation(), float(THETA))
    for m in messages:
        print(m)
    if ok:
        print(f"\nYour verification value for theta = {float(THETA)}: {value:.3f}")
        print("Type this value into Canvas question 1.")
    else:
        print("\nNot passed yet: no verification value. Fix build_rotation() and run both cells again.")


## Step 6 (optional now, needed in Module 5): connect your IBM Quantum account

You do not need real hardware until Module 5, but connecting now means it will be ready. Follow the page **Set up your tools** in Canvas first: it shows how to create a free IBM Quantum account, an API key and an instance.

**Keep your API key private.** Do not type it into a cell. In Colab, open **Secrets** (the key icon in the left sidebar) and add two secrets:

- `IBM_QUANTUM_API_KEY`: your API key;
- `IBM_QUANTUM_INSTANCE`: your instance CRN.

Turn on **Notebook access** for both, then run the cell. It only lists the quantum computers you can use; it does not use any of your free QPU time.

In [ ]:
try:
    from google.colab import userdata
    from qiskit_ibm_runtime import QiskitRuntimeService

    service = QiskitRuntimeService(token=userdata.get("IBM_QUANTUM_API_KEY"),
                                   instance=userdata.get("IBM_QUANTUM_INSTANCE"))
    names = [b.name for b in service.backends()]
    print("Connected. Quantum computers available to you:", ", ".join(names))
except Exception as exc:
    print("Not connected yet:", type(exc).__name__, "-", exc)
    print("Check the two secrets and their Notebook access, then run this cell again.")

## Finish

1. Answer the five questions of the **Module 0 lab check** in Canvas and submit.
2. Fill in the short **Module 0 time log**: how many minutes you spent working, and how many waiting (for installs or queues).

Save your work with **File > Save a copy in Drive** if you have not already.